### **Enviornment**

In [ ]:
# Setting up the local imports and env variables
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGCHAIN_TRACING_V2'] = os.getenv("LANGCHAIN_TRACING_V2")
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

### **Retriever**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma 
from langchain_openai import OpenAIEmbeddings

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250,chunk_overlap=0)
doc_splits = text_splitter.split_documents(docs_list)

import os 
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

# Add to VectorDB
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding=OpenAIEmbeddings()
)

retriver = vectorstore.as_retriever()

### **LLMs**

In [ ]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field 
from langchain_openai import ChatOpenAI

# Data model 
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrived documents."""
    binary_score: str = Field(
        description="Whether the documents are relevant to the question."
    )

#LLM with function call
llm = ChatOpenAI(model = "gpt-4o-mini" , temperature=0)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# Prompt 
system = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system" , system),
        ("human" , "Retrived document: \n\n {document} \n\n User Question : {question}")
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader
question = "agent_memory"
docs = retriver.invoke(question)
doc_txt = docs[1].page_content
print(retrieval_grader.invoke({"question" : question , "document" : doc_txt}))

In [28]:
from langchain_core.output_parsers import StrOutputParser

# Generate
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant.

Use ONLY the following context to answer the question.
If the answer cannot be found in the context, say "I don't know."

Context:
{documents}

Question:
{question}

Answer:
""")

llm = ChatOpenAI(model="gpt-4o-mini" , temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = prompt | llm | StrOutputParser()

generation = rag_chain.invoke({"documents" : docs , "question" : question})
print(generation)

The context mentions two types of memory in a LLM-powered autonomous agent system:

1. **Short-term memory**: This is utilized for in-context learning, allowing the model to learn from the immediate context.
2. **Long-term memory**: This enables the agent to retain and recall information over extended periods, often using an external vector store for fast retrieval.

Additionally, there is a mention of a "memory stream," which is a long-term memory module that records a comprehensive list of agents’ experiences in natural language.


In [ ]:
# Hallucination Grader
class GradeHallucination(BaseModel):
    """Binary score for hallucination present in the generated answer."""
    binary_score: str = Field(description="Answer is grounded in the facts, 'yes' or 'no'")

# LLM with function call
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)
structured_llm_grader = llm.with_structured_output(GradeHallucination)

# Prompt
system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
     Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""
hallucination_prompt = ChatPromptTemplate.from_messages({
    ('system' , system),
    ('human' , "Set of facts : \n\n{documents} \n\n LLM Generation : {generation}")
})

hallucination_grader = hallucination_prompt | structured_llm_grader
print(hallucination_grader.invoke({"documents" : docs , "generation" : generation}))

In [ ]:
# Answer Grader

class GradeAnswer(BaseModel):
    """Binary score to assess answer addresses questions"""
    binary_score: str = Field(description="Answer addresses the question, 'yes' or 'no'")

# LLM
llm = ChatOpenAI(model = "gpt-4o-mini" , temperature=0)
structured_llm_grader = llm.with_structured_output(GradeAnswer)

# Prompt 
system = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question."""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ('system' , system),
        ('human' , 'User Question : \n\n {question} \n\n LLM Generation : {generation}')
    ]
)

answer_grader = answer_prompt | structured_llm_grader
print(answer_grader.invoke({"question" : question , "generation" : generation}))

In [ ]:
# Question Re writer 

system = """You are a question re-writer that converts an input question to a better version that is optimized \n 
     for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning."""
re_write_prompt = ChatPromptTemplate.from_messages([
    ('system' , system),
    ('human' , 'Here is the initial question : \n\n{question} \n Formulate an improved question.')
])

question_retriver = re_write_prompt | llm | StrOutputParser()
question_retriver.invoke({"question" : question})

### **Graph**

In [ ]:
from typing import TypedDict 
from typing import List 

class GraphState(TypedDict):
    """Represent the state of our graph
       Args:
            question : question
            generation : LLM generation
            documents :list of documents"""
    question : str 
    generation : str 
    documents : List[str]

In [ ]:
# Nodes
from langchain_core.documents import Document
import pprint


def retrive(state):
    """
        Retrive the documents.
        Args:
            state(dict) : Current Graph State
        Returns:
            state(dict) : Return updateed graph state with document added as key init along with docs
    """
    print("---Retrive-----")
    question = state['question']

    # Retrival
    docs = retriver.invoke(question)
    return {"documents" : docs , "question" : question}


def generate(state):
    """ 
        Generate Answer
        Args:
            state(dict) : Current Graph State
        Returns:
            state(dict) : Return updated graph state with generation key value added in it
    """ 
    print("----Generate-----")
    question = state['question']
    documents = state['documents']

    # Generate Answer
    generation = rag_chain.invoke({"question" : question , "documents" : documents})
    return {"documents": documents, "question": question, "generation": generation}


def grade_documents(state):
    """ 
        Determine wheather the retrived content is relevant to the question
        Args: 
            state(dict) : current Graph State
        Returns:
            state(dict) : Update the documents key with only filtered relevant documents
    """
    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]

    # Score for each docs
    filtered_docs = []
    for d in documents:
        score = retrieval_grader.invoke({"question" : question , "document" : d.page_content})
        grade = score.binary_score
        if grade == "yes":
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(d)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            continue

    return {"documents": filtered_docs, "question": question}


def transform_query(state):
    """ 
        Transform the query to produce a better question
        Args:
            state(dict) : Current Graph State
        Returns:
            state(dict) : Updates question key with a re-phrased question
    """
    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]

    # Re-write question
    better_question = question_retriver.invoke({"question": question})
    return {"documents": documents, "question": better_question}


# Edges

def decide_to_generate(state):
    """ 
        Determine wheather to generate answer or regenarate the question
        Args:
            state(dict) : The Current Graph state
        Returns:
            str: Binary decision for next node to call
    """
    print("---ASSESS GRADED DOCUMENTS---")
    question = state["question"]
    filtered_documents = state["documents"]

    if not filtered_documents:
        # We will regenerate the query
        print("---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---")
        return "transform_query"
    else:
        # We have relevant documents, so generate answer
        print("---DECISION: GENERATE---")
        return "generate"
    

def grade_generation_v_documents_and_question(state):
    """
    Determines whether the generation is grounded in the document and answers question.
    Args:
        state (dict): The current graph state
    Returns:
        str: Decision for next node to call
    """

    print("---CHECK HALLUCINATIONS---")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]

    score = hallucination_grader.invoke({"documents": documents, "generation": generation})
    grade = score.binary_score

    # Check hallucination
    if grade == "yes":
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")
        # Check question-answering
        print("---GRADE GENERATION vs QUESTION---")
        score = answer_grader.invoke({"question": question,"generation": generation})
        grade = score.binary_score
        if grade == "yes":
            print("---DECISION: GENERATION ADDRESSES QUESTION---")
            return "useful"
        else:
            print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION---")
            return "not useful"
    else:
        pprint("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"


### **Graph Construction**

In [ ]:
from langgraph.graph import END,StateGraph

workflow = StateGraph(GraphState)

# Define the nodes
workflow.add_node("retrieve" , retrive)
workflow.add_node("grade_documents" , grade_documents)
workflow.add_node("generate",generate)
workflow.add_node("transform_query" , transform_query)

# Build graph
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve","grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "generate": "generate"
    }
)
workflow.add_edge("transform_query", "retrieve")
workflow.add_conditional_edges(
    "generate", 
    grade_generation_v_documents_and_question,
    {
        "not supported": "generate",
        "useful": END,
        "not useful": "transform_query",
    }
)

app = workflow.compile()

In [ ]:
png = app.get_graph().draw_mermaid_png()

with open("graph.png", "wb") as f:
    f.write(png)

In [30]:
from pprint import pprint

# Run
inputs = {"question": "Explain how the different types of agent memory work?"}
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
        # Optional: print full state at each node
        # pprint.pprint(value["keys"], indent=2, width=80, depth=None)
    pprint("\n---\n")

# Final generation
pprint(value["generation"])

---Retrive-----
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
"Node 'grade_documents':"
'\n---\n'
----Generate-----
---CHECK HALLUCINATIONS---
---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
---GRADE GENERATION vs QUESTION---
---DECISION: GENERATION ADDRESSES QUESTION---
"Node 'generate':"
'\n---\n'
('In a LLM-powered autonomous agent system, there are two main types of '
 'memory: short-term memory and long-term memory.\n'
 '\n'
 '1. **Short-term memory**: This is utilized through in-context learning, '
 'where the model learns from the immediate context provided in the prompts. '
 'It allows the agent to retain information temporarily to make decisions or '
 'perform tasks based on recent inputs.\n'
 '\n'
 '2. **Long-term memory**: This type of memory enables the agent to re

In [29]:
inputs = {"question": "Explain how chain of thought prompting works?"}
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
        # Optional: print full state at each node
        # pprint.pprint(value["keys"], indent=2, width=80, depth=None)
    pprint("\n---\n")

# Final generation
pprint(value["generation"])

---Retrive-----
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
"Node 'grade_documents':"
'\n---\n'
----Generate-----
---CHECK HALLUCINATIONS---
---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
---GRADE GENERATION vs QUESTION---
---DECISION: GENERATION ADDRESSES QUESTION---
"Node 'generate':"
'\n---\n'
('Chain of thought (CoT) prompting works by instructing the model to "think '
 'step by step" when tackling complex tasks. This technique enhances the '
 "model's performance by decomposing hard tasks into smaller, more manageable "
 'steps. By transforming larger tasks into multiple simpler tasks, CoT not '
 'only aids in the execution of the task but also provides insight into the '
 "model's reasoning process. This method allows the model to utilize more "
 'test-time computation, t